# MVP - Data Engineering
## Camada bronze

In [0]:
# 1. Python Libraries

from pyspark.sql.functions import col, when, isnull, sum as spark_sum, count as count_all, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType, BooleanType
from pyspark.sql.window import Window

In [0]:
%sql
-- Estabelecendo o catálogo usado no MVP (SQL)

USE CATALOG mvp_pucrio;

In [0]:
# Estabelecendo as variáveis para catálogo e schema para serem usadas neste Notebook em Pyspark.

catalog = "mvp_pucrio"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

Para garantir governança e qualidade dos dados, schemas explícitos foram definidos para cada a camada bronze. Isso garante validação dos dados desde o início do pipeline de dados, melhor performance (sem necessidade de inferir tipos), e melhor documentação dos dados.

In [0]:
# 2. Schemas explícitos para a camada bronze

schemas = {
    "customers": StructType([
        StructField("customer_id", StringType(), False),
        StructField("customer_unique_id", StringType(), True),
        StructField("customer_zip_code_prefix", IntegerType(), True),
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True)
    ]),
    
    "orders": StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType(), False),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True)
    ]),
    
    "order_items": StructType([
        StructField("order_id", StringType(), False),
        StructField("order_item_id", IntegerType(), True),
        StructField("product_id", StringType(), False),
        StructField("seller_id", StringType(), False),
        StructField("shipping_limit_date", TimestampType(), True),
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True)
    ]),
    
    "products": StructType([
        StructField("product_id", StringType(), False),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", IntegerType(), True),
        StructField("product_length_cm", IntegerType(), True),
        StructField("product_height_cm", IntegerType(), True),
        StructField("product_width_cm", IntegerType(), True)
    ]),
    
    "sellers": StructType([
        StructField("seller_id", StringType(), False),
        StructField("seller_zip_code_prefix", IntegerType(), True),
        StructField("seller_city", StringType(), True),
        StructField("seller_state", StringType(), True)
    ]),
    
    "marketing_qualified_leads": StructType([
        StructField("mql_id", StringType(), False),
        StructField("first_contact_date", DateType(), True),
        StructField("landing_page_id", StringType(), True),
        StructField("origin", StringType(), True)
    ]),
    
    "closed_deals": StructType([
        StructField("mql_id", StringType(), False),
        StructField("seller_id", StringType(), True),
        StructField("sdr_id", StringType(), True),
        StructField("sr_id", StringType(), True),
        StructField("won_date", TimestampType(), True),
        StructField("business_segment", StringType(), True),
        StructField("lead_type", StringType(), True),
        StructField("lead_behaviour_profile", StringType(), True),
        StructField("has_company", BooleanType(), True),
        StructField("has_gtin", BooleanType(), True),
        StructField("average_stock", StringType(), True),
        StructField("business_type", StringType(), True),
        StructField("declared_product_catalog_size", DoubleType(), True),
        StructField("declared_monthly_revenue", DoubleType(), True)
    ])
}

As tabelas da camada bronze são nomeadas com base nos arquivos em "raw_files/csv_files", removendo prefixos ("olist_") e sufixos ("_dataset.csv"). Por exemplo, o arquivo "olist_customers_dataset.csv" será chamado de "customers" no schema bronze.

Durante a ingestão, cada tabela Bronze recebe duas colunas de controle não presentes na fonte original: _source_file (nome do arquivo CSV de origem) e _ingested_at (timestamp da carga). Essas colunas existem apenas na camada Bronze e servem para rastreabilidade, pois permitem identificar de qual carga e arquivo cada registro veio caso seja necessário investigar um problema. Elas são removidas na Silver pela função removing_cols.

Para "escrever" as tabelas na camada bronze, o modo overwrite foi utilizado, mas em um ambiente de desenvolvimento, é provável que usaria "incremental"/"merge" para adicionar novos dados ao invés de escrever tudo de novo.

In [0]:
# 3. Lendo dados do Volume do Unity Catalog (UC) com schemas explícitos

volume_path = "/Volumes/mvp_pucrio/raw_files/csv_files"
catalog_prefix = f"{catalog}.bronze."

files = dbutils.fs.ls(volume_path)

for fl in files:
    fl = fl.name
    if fl.endswith('.csv'):
        fl_name = fl.split('.')[0]
        fl_name = fl_name.replace('olist_', '').replace('_dataset', '')
        
        schema = schemas.get(fl_name)
        if schema is None:
            print(f"- Schema não encontrado para {fl_name}, pulando arquivo...")
            continue
        
        df_orders = spark.read.csv(
            f"{volume_path}/{fl}",
            header=True,
            schema=schema,
            multiLine=True
        )
        
        df_orders = (df_orders
            .withColumn("_source_file", lit(fl))
            .withColumn("_ingested_at", current_timestamp())
        )

        df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            catalog_prefix + fl_name
        )
        
        print(f"- {fl_name} carregado com schema explícito")

## Documentando as tabelas
Abaixo estou documentando todas as tabelas.

In [0]:
%sql
-- Configurando contexto para documentação

USE SCHEMA bronze;

In [0]:
%sql
-- Documentando tabela customers

COMMENT ON TABLE customers IS 
'Dados de clientes que realizaram compras na plataforma. Cada registro representa um cliente único identificado por customer_id.';

COMMENT ON COLUMN customers.customer_id IS 
'ID único do cliente (PK)';

COMMENT ON COLUMN customers.customer_unique_id IS 
'ID único universal do cliente - usado para identificar o mesmo cliente em múltiplos pedidos';

COMMENT ON COLUMN customers.customer_zip_code_prefix IS 
'Prefixo do CEP do cliente (primeiros 5 dígitos)';

COMMENT ON COLUMN customers.customer_city IS 
'Cidade de residência do cliente';

COMMENT ON COLUMN customers.customer_state IS 
'Estado (UF) de residência do cliente';

In [0]:
%sql
-- Documentando tabela orders

COMMENT ON TABLE orders IS 
'Pedidos realizados pelos clientes. Contém informações de status e timestamps do ciclo de vida do pedido (compra, aprovação, envio, entrega).';

COMMENT ON COLUMN orders.order_id IS 
'ID único do pedido (PK)';

COMMENT ON COLUMN orders.customer_id IS 
'ID do cliente que realizou o pedido (FK para customers)';

COMMENT ON COLUMN orders.order_status IS 
'Status atual do pedido (delivered, shipped, canceled, etc)';

COMMENT ON COLUMN orders.order_purchase_timestamp IS 
'Data e hora da compra';

COMMENT ON COLUMN orders.order_approved_at IS 
'Data e hora da aprovação do pagamento';

COMMENT ON COLUMN orders.order_delivered_carrier_date IS 
'Data e hora da entrega ao transportador';

COMMENT ON COLUMN orders.order_delivered_customer_date IS 
'Data e hora da entrega ao cliente';

COMMENT ON COLUMN orders.order_estimated_delivery_date IS 
'Data estimada de entrega informada ao cliente';

In [0]:
%sql
-- Documentando tabela order_items

COMMENT ON TABLE order_items IS 
'Itens individuais de cada pedido. Um pedido pode conter múltiplos itens. Relaciona pedidos com produtos e sellers.';

COMMENT ON COLUMN order_items.order_id IS 
'ID do pedido (PK composta, FK para orders)';

COMMENT ON COLUMN order_items.order_item_id IS 
'Número sequencial do item dentro do pedido';

COMMENT ON COLUMN order_items.product_id IS 
'ID do produto (PK composta, FK para products)';

COMMENT ON COLUMN order_items.seller_id IS 
'ID do vendedor (PK composta, FK para sellers)';

COMMENT ON COLUMN order_items.shipping_limit_date IS 
'Data limite para o seller enviar o produto ao transportador';

COMMENT ON COLUMN order_items.price IS 
'Preço unitário do item (R$)';

COMMENT ON COLUMN order_items.freight_value IS 
'Valor do frete deste item (R$)';

In [0]:
%sql
-- Documentando tabela products

COMMENT ON TABLE products IS 
'Catálogo de produtos disponíveis na plataforma. Inclui características físicas e descritivas dos produtos.';

COMMENT ON COLUMN products.product_id IS 
'ID único do produto (PK)';

COMMENT ON COLUMN products.product_category_name IS 
'Categoria do produto (em português)';

COMMENT ON COLUMN products.product_name_lenght IS 
'Comprimento do nome do produto (número de caracteres)';

COMMENT ON COLUMN products.product_description_lenght IS 
'Comprimento da descrição do produto (número de caracteres)';

COMMENT ON COLUMN products.product_photos_qty IS 
'Quantidade de fotos do produto';

COMMENT ON COLUMN products.product_weight_g IS 
'Peso do produto (gramas)';

COMMENT ON COLUMN products.product_length_cm IS 
'Comprimento do produto (cm)';

COMMENT ON COLUMN products.product_height_cm IS 
'Altura do produto (cm)';

COMMENT ON COLUMN products.product_width_cm IS 
'Largura do produto (cm)';

In [0]:
%sql
-- Documentando tabela sellers

COMMENT ON TABLE sellers IS 
'Vendedores/fornecedores cadastrados na plataforma. Cada seller pode vender múltiplos produtos.';

COMMENT ON COLUMN sellers.seller_id IS 
'ID único do seller/vendedor (PK)';

COMMENT ON COLUMN sellers.seller_zip_code_prefix IS 
'Prefixo do CEP do seller (primeiros 5 dígitos)';

COMMENT ON COLUMN sellers.seller_city IS 
'Cidade onde o seller está localizado';

COMMENT ON COLUMN sellers.seller_state IS 
'Estado (UF) onde o seller está localizado';

In [0]:
%sql
-- Documentando tabela marketing_qualified_leads

COMMENT ON TABLE marketing_qualified_leads IS 
'Leads qualificados de marketing (MQLs). Prospects que demonstraram interesse e foram qualificados pelo time de marketing.';

COMMENT ON COLUMN marketing_qualified_leads.mql_id IS 
'ID único do lead qualificado de marketing (PK)';

COMMENT ON COLUMN marketing_qualified_leads.first_contact_date IS 
'Data do primeiro contato com o lead';

COMMENT ON COLUMN marketing_qualified_leads.landing_page_id IS 
'ID da landing page de origem do lead';

COMMENT ON COLUMN marketing_qualified_leads.origin IS 
'Canal de origem do lead (organic_search, paid_search, social, etc)';

In [0]:
%sql
-- Documentando tabela closed_deals

COMMENT ON TABLE closed_deals IS 
'Negócios fechados. Contém informações sobre leads que se converteram em sellers ativos na plataforma.';

COMMENT ON COLUMN closed_deals.mql_id IS 
'ID do lead (PK, FK para marketing_qualified_leads)';

COMMENT ON COLUMN closed_deals.seller_id IS 
'ID do seller resultante da conversão (FK para sellers)';

COMMENT ON COLUMN closed_deals.sdr_id IS 
'ID do SDR (Sales Development Representative) responsável';

COMMENT ON COLUMN closed_deals.sr_id IS 
'ID do SR (Sales Representative) responsável';

COMMENT ON COLUMN closed_deals.won_date IS 
'Data e hora do fechamento do negócio';

COMMENT ON COLUMN closed_deals.business_segment IS 
'Segmento de negócio do seller (pet, health_beauty, electronics, etc)';

COMMENT ON COLUMN closed_deals.lead_type IS 
'Tipo/tamanho do lead (online_small, online_medium, online_big, etc)';

COMMENT ON COLUMN closed_deals.lead_behaviour_profile IS 
'Perfil comportamental do lead durante o processo de vendas';

COMMENT ON COLUMN closed_deals.has_company IS 
'Indica se o seller possui CNPJ';

COMMENT ON COLUMN closed_deals.has_gtin IS 
'Indica se o seller possui código GTIN nos produtos';

COMMENT ON COLUMN closed_deals.average_stock IS 
'Estoque médio declarado pelo seller';

COMMENT ON COLUMN closed_deals.business_type IS 
'Tipo de negócio (reseller, manufacturer, etc)';

COMMENT ON COLUMN closed_deals.declared_product_catalog_size IS 
'Tamanho do catálogo de produtos declarado';

COMMENT ON COLUMN closed_deals.declared_monthly_revenue IS 
'Receita mensal declarada (R$)';

## Verificando as tabelas

Abaixo, uma rapida verificacao de cada tabela para checar possiveis ajustes necessarios na camada silver.

In [0]:
%sql
-- Verificando tabela customers

SELECT * 
FROM customers 
ORDER BY customer_id
LIMIT 10;

In [0]:
%sql
-- Verificando tabela orders

SELECT * 
FROM orders 
ORDER BY order_id
LIMIT 10;

In [0]:
%sql
-- Verificando tabela order_items

SELECT * 
FROM order_items 
ORDER BY order_id
LIMIT 10;

In [0]:
%sql
-- Verificando tabela products

SELECT * 
FROM products 
ORDER BY product_id
LIMIT 10;

In [0]:
%sql
-- Verificando tabela sellers

SELECT *
FROM sellers
ORDER BY seller_id
LIMIT 10;

In [0]:
%sql
-- Verificando tabela marketing_qualified_leads

SELECT * 
FROM marketing_qualified_leads 
ORDER BY mql_id
LIMIT 10;

In [0]:
%sql
-- Verificando tabela closed_deals

SELECT * 
FROM closed_deals 
ORDER BY mql_id
LIMIT 10;